# C. elegans — Connectivity Notebook: fastC-GC*

Prerequisite: the dataset `shared/windows` checkpoint must exist.
Parallel lane: yes. This notebook owns only its method namespace under `outputs/analysis/<DATASET_ID>/<RECORDING_ID>/<RUN_ID>/<METHOD>/`.
Execution: one model is fit per window. `CHECKPOINT_CHUNK_SIZE` controls how many fits share an atomic resume checkpoint; it is not the total fit count.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [1]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


Notebook interpreter: /Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/bin/python
Installed backends: {'fastcgc': '0.1.0', 'tigramite': '5.2.10.1', 'scikit-learn': '1.9.0', 'kmedoids': '0.5.5', 'igraph': '1.0.0', 'leidenalg': '0.12.0'}


In [2]:
from omegaconf import OmegaConf

from effectome.connectivity import ConnectivityConfig
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_connectivity,
    stage_artifact_path,
)

DATASET_ID = "c_elegans"
DATASET_OUTPUT_ID = "bundle-net-c-elegans"
RECORDING_ID = "bundle-net-worm-0"
RUN_ID = "reference"
METHOD = "cgc_star"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
WINDOWS_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "windows" / "artifact.pkl"
CONNECTIVITY_CONFIG_PATH = PROJECT_ROOT / "conf" / "connectivity" / "cgc_star.yaml"
FORCE = False
CHECKPOINT_CHUNK_SIZE = 16

CONNECTIVITY_YAML = OmegaConf.to_container(
    OmegaConf.load(CONNECTIVITY_CONFIG_PATH), resolve=True
)
if not isinstance(CONNECTIVITY_YAML, dict):
    raise TypeError(f"Expected a mapping in {CONNECTIVITY_CONFIG_PATH}")
CONNECTIVITY_CFG = ConnectivityConfig(**CONNECTIVITY_YAML)

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} connectivity",
    expected_stages=["windows_input", "connectivity"],
    force=FORCE,
)


/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cgc_star connectivity
run_id=reference method=cgc_star
run_dir=/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/analysis/bundle-net-c-elegans/bundle-net-worm-0/reference/cgc_star
resume mode: FORCE=False, matching completed checkpoints will be reused
Rerun interrupted notebooks from the top; completed stages resume from checkpoints.
Stage status for reference/cgc_star:
  - connectivity: not started
  - windows_input: not started


In [3]:
connectivity = run_connectivity(
    run,
    windows_path=WINDOWS_PATH,
    connectivity_cfg=CONNECTIVITY_CFG,
    chunk_size=CHECKPOINT_CHUNK_SIZE,
    force=FORCE,
)
print(connectivity.matrices.shape)
print(connectivity.diagnostics["runtime"])
print(stage_artifact_path(run, "connectivity"))
print_stage_status(run)


[reference/cgc_star] windows_input: checking checkpoint; computing if needed...
[reference/cgc_star] windows_input: completed in 0.2s
[reference/cgc_star] connectivity plan: 186 independent window fits in 12 resumable checkpoint chunks (up to 16 fits per chunk)
[reference/cgc_star] connectivity: checking checkpoint; computing if needed...


Connectivity fits: 100%|██████████| 186/186 [14:28<00:00,  4.67s/fit]

[reference/cgc_star] connectivity: completed in 951.1s
(186, 109, 109)
{'elapsed_seconds': 82.81087191399911, 'seconds_per_window': 5.175679494624944, 'dense_output_bytes': 1520768, 'note': 'Output memory only; estimator workspace and NumPy-native allocations are not included.'}
/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/analysis/bundle-net-c-elegans/bundle-net-worm-0/reference/cgc_star/stages/connectivity/artifact.pkl
Stage status for reference/cgc_star:
  - connectivity: completed (signature=6918ffd92f, artifact=yes)
  - connectivity.chunk-00000: completed (signature=3f5ca58960, artifact=yes)
  - connectivity.chunk-00001: completed (signature=28cfd3a16b, artifact=yes)
  - connectivity.chunk-00002: completed (signature=af336ee3d3, artifact=yes)
  - connectivity.chunk-00003: completed (signature=4526b4361f, artifact=yes)
  - connectivity.chunk-00004: completed (signature=58a6fbce76, artifact=yes)
  - connectivity.chunk-00005: completed (s